In [1]:
!nvidia-smi

Fri Feb 20 16:41:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40                     Off |   00000000:00:10.0 Off |                    0 |
| N/A   37C    P8             35W /  300W |      14MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [26]:
import numpy as np
import faiss
import ollama
import pickle
import time
from sklearn.cluster import KMeans
import csv
import sys


# --- CONFIGURATION ---
DB_FILE = '../vector_db_oran_all.pkl'
EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
GENERATION_MODEL = 'mistral' 
N_CLUSTERS = 5 


# Load Data
with open(DB_FILE, 'rb') as f:
    VECTOR_DB = pickle.load(f)

embeddings = np.array([item['embedding'] for item in VECTOR_DB]).astype('float32')
dimension = embeddings.shape[1]

# --- 1. SETUP HIERARCHIES ---

# Manual setup (K-Means Centroids)
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10).fit(embeddings)
centroids = kmeans.cluster_centers_
cluster_labels = kmeans.labels_

# FAISS setup (Inverted File Index)
quantizer = faiss.IndexFlatL2(dimension)
index = faiss.IndexIVFFlat(quantizer, dimension, N_CLUSTERS)
index.train(embeddings)
index.add(embeddings)
index.nprobe = 1

# --- 2. RETRIEVAL & TIMING FUNCTIONS ---

def get_manual_knowledge(query_emb):
    start = time.perf_counter()
    # Path: Query -> Closest Centroid -> Local Search
    centroid_id = np.argmin(np.linalg.norm(centroids - query_emb, axis=1))
    indices = [i for i, l in enumerate(cluster_labels) if l == centroid_id]
    cluster_embs = embeddings[indices]
    sub_dists = np.linalg.norm(cluster_embs - query_emb, axis=1)
    best_leaf_indices = [indices[i] for i in np.argsort(sub_dists)[:3]]
    latency = (time.perf_counter() - start) * 1000
    return [VECTOR_DB[i]['content'] for i in best_leaf_indices], centroid_id, latency

def get_faiss_knowledge(query_emb):
    q_emb_f = np.array([query_emb]).astype('float32')
    start = time.perf_counter()
    # FAISS performs the hierarchical jump in C++
    _, indices = index.search(q_emb_f, k=3)
    latency = (time.perf_counter() - start) * 1000
    return [VECTOR_DB[i]['content'] for i in indices[0] if i != -1], latency

def run_generation(query, context, method_name):
    if not GENERATION_MODEL: return "[Generation skipped]"
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    return ollama.generate(model=GENERATION_MODEL, prompt=prompt)['response']

# --- 3. EXECUTION ---


def run_generation(query, context):
    """
    Uses a strict system prompt to force the model to behave.
    """
    system_instruction = (
        "You are a multiple-choice answering engine. "
        "Your goal is to provide ONLY the correct letter (A, B, C, or D) "
        "and a single short sentence explaining why. "
        "Do not say 'The correct answer is', do not say 'Based on the context'. "
        "Format: [Letter]"
    )
    
    user_prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    response = ollama.generate(
        model=GENERATION_MODEL, 
        system=system_instruction,
        prompt=user_prompt,
        options={"temperature": 0.0} # Set to 0 for deterministic, objective answers
    )
    
    # Post-processing: remove any residual junk like "Answer: " or newlines
    clean_text = response['response'].strip().replace('\n', ' ')
    return clean_text

# --- IMPROVED BATCH PROCESSING ---


def process_qa_batch(input_csv_path, output_csv_path):
    output_headers = ["Question", "FAISS Response", "Manual Response", "Latency FAISS", "Latency Manual"]

    try:
        with open(input_csv_path, mode='r', encoding='utf-8') as infile:
            reader = csv.DictReader(infile)
            
            with open(output_csv_path, mode='w', encoding='utf-8', newline='') as outfile:
                # csv.QUOTE_NONNUMERIC wraps all text in quotes to stop commas from breaking rows
                writer = csv.DictWriter(outfile, fieldnames=output_headers, quoting=csv.QUOTE_NONNUMERIC)
                writer.writeheader()

                for i, row in enumerate(reader):
                    full_query = (
                        f"{row['Question']}\n"
                        f"A) {row['A']} B) {row['B']} C) {row['C']} D) {row['D']}"
                    )
                    
                    # 1. Embed
                    q_emb = np.array(ollama.embed(model=EMBEDDING_MODEL, input=full_query)['embeddings'][0])

                    # 2. Retrieve
                    man_docs, cluster_id, man_lat = get_manual_knowledge(q_emb)
                    fai_docs, fai_lat = get_faiss_knowledge(q_emb)

                    # 3. Generate (Cleaning is handled inside the function)
                    ans_man = run_generation(full_query, "\n".join(man_docs))
                    ans_fai = run_generation(full_query, "\n".join(fai_docs))

                    # 4. Final Formatting
                    writer.writerow({
                        "Question": row['Question'],
                        "FAISS Response": ans_fai,
                        "Manual Response": f"[Cluster {cluster_id}] {ans_man}",
                        "Latency FAISS": f"{fai_lat:.4f} ms (FAISS Execution)",
                        "Latency Manual": f"{man_lat:.4f} ms"
                    })
                    print(f"Row {i+1} saved.")

    except Exception as e:
        print(f"\n❌ ERROR at row {i+1 if 'i' in locals() else '1'}: {e}")
        raise

In [27]:
process_qa_batch('input_questions.csv', 'output_responses.csv')

Row 1 saved.
Row 2 saved.
Row 3 saved.
Row 4 saved.
Row 5 saved.
Row 6 saved.
Row 7 saved.
Row 8 saved.
Row 9 saved.
Row 10 saved.
Row 11 saved.
Row 12 saved.
Row 13 saved.
Row 14 saved.
Row 15 saved.
Row 16 saved.
Row 17 saved.
Row 18 saved.
Row 19 saved.
Row 20 saved.
Row 21 saved.
Row 22 saved.
Row 23 saved.
Row 24 saved.
Row 25 saved.
Row 26 saved.
Row 27 saved.
Row 28 saved.
Row 29 saved.
Row 30 saved.
Row 31 saved.
Row 32 saved.
Row 33 saved.
Row 34 saved.
Row 35 saved.
Row 36 saved.
Row 37 saved.
Row 38 saved.
Row 39 saved.
Row 40 saved.
Row 41 saved.
Row 42 saved.
Row 43 saved.
Row 44 saved.
Row 45 saved.
Row 46 saved.
Row 47 saved.
Row 48 saved.
Row 49 saved.
Row 50 saved.
Row 51 saved.
Row 52 saved.
Row 53 saved.
Row 54 saved.
Row 55 saved.
Row 56 saved.
Row 57 saved.
Row 58 saved.
Row 59 saved.
Row 60 saved.
Row 61 saved.
Row 62 saved.
Row 63 saved.
Row 64 saved.
Row 65 saved.
Row 66 saved.
Row 67 saved.
Row 68 saved.
Row 69 saved.
Row 70 saved.
Row 71 saved.
Row 72 saved.
R

In [31]:
import pandas as pd
import re

def extract_choice_letter(text):
    """
    Extracts a choice letter (A-D) from a string, 
    even if it's inside brackets [A] or followed by a parenthesis A).
    """
    if not isinstance(text, str):
        return None
    
    # Matches a single capital letter inside [] or before )
    # This specifically targets patterns like '[A]' or 'C)' 
    # and ignores prefixes like '[Cluster 0]'
    match = re.search(r'\[([A-D])\]|([A-D])\)', text)
    
    if match:
        # Group 1 captures [A], Group 2 captures A)
        return match.group(1) if match.group(1) else match.group(2)
    return None

def process_oran_data(input_file, output_file):
    # Read the dataset
    df = pd.read_csv(input_file)
    
    # Create new columns by applying the extraction function
    df['FAISS_Letter'] = df['FAISS Response'].apply(extract_choice_letter)
    df['Manual_Letter'] = df['Manual Response'].apply(extract_choice_letter)
    
    # Save the updated DataFrame to a new CSV file
    df.to_csv(output_file, index=False)
    print(f"Extraction complete. Results saved to '{output_file}'.")

if __name__ == "__main__":
    # Define file names
    SOURCE_FILE = './output_responses.csv'
    TARGET_FILE = 'newOutput.csv'
    
    try:
        process_oran_data(SOURCE_FILE, TARGET_FILE)
    except FileNotFoundError:
        print(f"Error: Could not find '{SOURCE_FILE}'. Please ensure the file exists.")

Extraction complete. Results saved to 'newOutput.csv'.
